# <font color=#0099CC>**Cross-Model Comparison**</font>

Scans `11_Portfolio/{run_tag}/` subfolders, loads results, and produces:

1. Summary table — avg returns across offsets for all strategies + benchmarks.
2. Cumulative return overlay — all models on one chart per strategy.
3. Bar chart — avg returns grouped by model.
4. Prediction quality — directional accuracy, expected vs actual.

**No TensorFlow required** — reads CSV/JSON only.

---
## <font color=#0099CC>**1. ENVIRONMENT**</font>

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

def detect_env():
    try:
        import google.colab
        return 'colab'
    except ImportError:
        return 'local'

ENV = detect_env()
if ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Taller4_DL_MIAX'
else:
    BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))

PORTFOLIO_DIR = os.path.join(BASE, '11_Portfolio')
print(f'> Entorno: {ENV}\n> PORTFOLIO_DIR: {PORTFOLIO_DIR}')

COMP_DIR = os.path.join(PORTFOLIO_DIR, 'Comparison_Files')
os.makedirs(COMP_DIR, exist_ok=True)
print(f'> COMP_DIR: {COMP_DIR}')


---
## <font color=#0099CC>**2. DISCOVER & LOAD ALL RESULTS**</font>

In [ ]:
def safe_read_csv(path, **kwargs):
    """Read CSV, return None if file is empty, missing, or corrupt."""
    if not os.path.isfile(path):
        return None
    if os.path.getsize(path) < 10:
        return None
    try:
        return pd.read_csv(path, **kwargs)
    except Exception:
        return None


def safe_read_json(path):
    """Read JSON, return None if file is missing or corrupt."""
    if not os.path.isfile(path):
        return None
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


models = {}
skipped = []

for folder in sorted(os.listdir(PORTFOLIO_DIR)):
    folder_path = os.path.join(PORTFOLIO_DIR, folder)
    if not os.path.isdir(folder_path):
        continue

    # Only process folders that look like run tags (inv_*)
    if not folder.startswith('inv_'):
        continue

    meta_path = os.path.join(folder_path, 'model_meta.json')
    meta = safe_read_json(meta_path)
    if meta is None:
        skipped.append((folder, 'no model_meta.json'))
        continue

    # Check required fields
    if 'avg_returns' not in meta:
        skipped.append((folder, 'no avg_returns in meta'))
        continue

    nav_df = safe_read_csv(
        os.path.join(folder_path, 'nav_series_offset0.csv'),
        index_col=0, parse_dates=True,
    )
    det_df = safe_read_csv(os.path.join(folder_path, 'window_detail.csv'))
    sum_df = safe_read_csv(os.path.join(folder_path, 'summary_returns.csv'))

    models[folder] = {
        'meta': meta,
        'nav': nav_df,
        'detail': det_df,
        'summary': sum_df,
    }

print(f'Loaded {len(models)} model result(s):\n')
for tag, data in models.items():
    m = data['meta']
    avg = m.get('avg_returns', {})
    has_nav = data['nav'] is not None
    has_det = data['detail'] is not None
    print(f'  {tag}')
    print(f'    arch={m.get("arch","?")}  v_in={m.get("v_in","?")}  v_out={m.get("v_out","?")}')
    print(f'    MAE test={m.get("mae_test",0):.5f}  NAV={has_nav}  Detail={has_det}')
    if avg:
        print(f'    Avg: A={avg["A"]:+.2%}  B={avg["B"]:+.2%}  '
              f'C={avg["C"]:+.2%}  BM1={avg["BM1"]:+.2%}  BM2={avg["BM2"]:+.2%}')

if skipped:
    print(f'\nSkipped {len(skipped)} folder(s):')
    for name, reason in skipped:
        print(f'  {name}: {reason}')

if not models:
    raise RuntimeError('No model results found! Run 05_batch_runner.ipynb first.')

---
## <font color=#0099CC>**3. SUMMARY TABLE — All Models**</font>

In [ ]:
rows = []
for tag, data in sorted(models.items()):
    m = data['meta']
    avg = m.get('avg_returns', {})
    if not avg:
        continue
    rows.append({
        'Model': tag,
        'Arch': m.get('arch', '?'),
        'V_IN': m.get('v_in', '?'),
        'MAE (test)': m.get('mae_test', 0),
        'Strat A': avg.get('A', 0),
        'Strat B': avg.get('B', 0),
        'Strat C': avg.get('C', 0),
        'BM1 (Fixed)': avg.get('BM1', 0),
        'BM2 (Rebal)': avg.get('BM2', 0),
        'A vs BM1': avg.get('A', 0) - avg.get('BM1', 0),
        'C vs BM1': avg.get('C', 0) - avg.get('BM1', 0),
    })

comp_df = pd.DataFrame(rows).sort_values('Strat A', ascending=False)

# Display formatted
disp = comp_df.copy()
for c in ['Strat A','Strat B','Strat C','BM1 (Fixed)','BM2 (Rebal)','A vs BM1','C vs BM1']:
    disp[c] = disp[c].apply(lambda x: f'{x:+.2%}')
disp['MAE (test)'] = disp['MAE (test)'].apply(lambda x: f'{x:.5f}')

print('-- Cross-Model Comparison (avg across offsets) --')
print(disp.to_string(index=False))

In [ ]:
# ── Summary table as figure ──
fig, ax = plt.subplots(figsize=(16, 1.5 + 0.5 * len(comp_df)))
ax.axis('off')

col_labels = ['Model', 'Arch', 'V_IN', 'MAE', 'A', 'B', 'C', 'BM1', 'BM2', 'A-BM1', 'C-BM1']
table_data = []
for _, r in comp_df.iterrows():
    table_data.append([
        r['Model'], r['Arch'], str(int(r['V_IN'])),
        f'{r["MAE (test)"]:.4f}',
        f'{r["Strat A"]:+.1%}', f'{r["Strat B"]:+.1%}', f'{r["Strat C"]:+.1%}',
        f'{r["BM1 (Fixed)"]:+.1%}', f'{r["BM2 (Rebal)"]:+.1%}',
        f'{r["A vs BM1"]:+.1%}', f'{r["C vs BM1"]:+.1%}',
    ])

table = ax.table(cellText=table_data, colLabels=col_labels, cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.0, 1.6)

for j in range(len(col_labels)):
    table[0, j].set_facecolor('#0099CC')
    table[0, j].set_text_props(color='white', fontweight='bold')
for i in range(len(table_data)):
    c = '#F5F5F5' if i % 2 == 0 else '#FFFFFF'
    for j in range(len(col_labels)):
        table[i+1, j].set_facecolor(c)

ax.set_title('Cross-Model Performance Comparison (avg returns across offsets)', fontsize=13, pad=20)
plt.tight_layout()
plt.savefig(os.path.join(COMP_DIR, 'comparison_table.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## <font color=#0099CC>**4. CUMULATIVE RETURN OVERLAY — All Models per Strategy**</font>

In [ ]:
arch_colors = {'cnn':'#2196F3','conv1d':'#2196F3','mlp':'#E91E63','rnn':'#FF9800','mixto':'#4CAF50'}

# Only models with NAV data
models_with_nav = {t: d for t, d in models.items() if d['nav'] is not None}

if models_with_nav:
    strat_keys = [('A', 'Strategy A (Equal Weight)'),
                  ('B', 'Strategy B (Take-Profit)'),
                  ('C', 'Strategy C (Proportional)')]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, (sk, stitle) in zip(axes, strat_keys):
        for tag, data in sorted(models_with_nav.items()):
            nav = data['nav']
            # Find the column that matches this strategy key
            col = None
            for c in nav.columns:
                if c == sk or c.startswith(f'Strategy {sk}') or c.startswith(f'Strat {sk}'):
                    col = c
                    break
            if col is None:
                continue
            arch = data['meta'].get('arch', '?')
            color = arch_colors.get(arch, '#888888')
            ax.plot(nav.index, (nav[col]-1)*100, label=tag, color=color, lw=1.5, alpha=0.8)

        # Add BM1 reference from first model
        first_tag = list(models_with_nav.keys())[0]
        bm1_nav = models_with_nav[first_tag]['nav']
        bm1_col = None
        for c in bm1_nav.columns:
            if 'BM1' in c:
                bm1_col = c
                break
        if bm1_col:
            ax.plot(bm1_nav.index, (bm1_nav[bm1_col]-1)*100,
                    color='#9E9E9E', ls='--', lw=1.5, label='BM1', alpha=0.6)

        ax.axhline(0, color='black', lw=0.5, alpha=0.3)
        ax.set_title(stitle, fontsize=11)
        ax.set_ylabel('Return (%)')
        ax.legend(fontsize=7, loc='best')
        ax.grid(True, alpha=0.2)

    plt.suptitle('Cumulative Returns by Strategy — All Models (offset=0)', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(COMP_DIR, 'comparison_cumulative.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No NAV data available for charting.')

---
## <font color=#0099CC>**5. BAR CHART — Average Returns**</font>

In [ ]:
model_tags = list(comp_df['Model'])
strats = ['Strat A', 'Strat B', 'Strat C']
strat_labels = ['Strat A (Equal Wt)', 'Strat B (Take-Profit)', 'Strat C (Proportional)']
strat_colors = ['#2196F3', '#FF9800', '#4CAF50']

x = np.arange(len(model_tags))
width = 0.22

fig, ax = plt.subplots(figsize=(max(10, 2.5*len(model_tags)), 5))

for i, (sk, sl, sc) in enumerate(zip(strats, strat_labels, strat_colors)):
    vals = [comp_df.loc[comp_df['Model']==t, sk].values[0]*100 for t in model_tags]
    ax.bar(x + i*width, vals, width, label=sl, color=sc, alpha=0.85)

# BM1 line
bm1_vals = [comp_df.loc[comp_df['Model']==t, 'BM1 (Fixed)'].values[0]*100 for t in model_tags]
avg_bm1 = np.mean(bm1_vals)
ax.axhline(avg_bm1, color='#9E9E9E', ls='--', lw=2, label=f'BM1 avg ({avg_bm1:.1f}%)')

ax.set_xticks(x + width)
ax.set_xticklabels([t.replace('inv_','').replace('_','/') for t in model_tags],
                    rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Avg Total Return (%)')
ax.set_title('Average Total Returns by Model & Strategy', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2, axis='y')
ax.axhline(0, color='black', lw=0.5)

plt.tight_layout()
plt.savefig(os.path.join(COMP_DIR, 'comparison_bars.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## <font color=#0099CC>**6. PREDICTION QUALITY**</font>

In [ ]:
pred_rows = []

for tag, data in sorted(models.items()):
    det = data['detail']
    m = data['meta']

    if det is None or det.empty:
        # No selections made — model predicted all negative
        pred_rows.append({
            'Model': tag,
            'Arch': m.get('arch', '?'),
            'V_IN': m.get('v_in', '?'),
            'MAE (test)': m.get('mae_test', 0),
            'Dir Correct': '0/0',
            'Dir %': 0,
            'Avg Exp Port Ret': 0,
            'Avg Act Port Ret': 0,
            'Avg Pred Error': 0,
            'Note': 'No assets selected (all negative exp returns)',
        })
        continue

    n_correct = int(det['direction_correct'].sum())
    n_total = len(det)

    # Portfolio-level expected vs actual per window
    windows = det.groupby('window')
    exp_ports = []
    act_ports = []
    for _, grp in windows:
        exp_ports.append((grp['weight_eq'] * grp['exp_return']).sum())
        act_ports.append((grp['weight_eq'] * grp['actual_return']).sum())

    pred_rows.append({
        'Model': tag,
        'Arch': m.get('arch', '?'),
        'V_IN': m.get('v_in', '?'),
        'MAE (test)': m.get('mae_test', 0),
        'Dir Correct': f'{n_correct}/{n_total}',
        'Dir %': n_correct / n_total if n_total > 0 else 0,
        'Avg Exp Port Ret': np.mean(exp_ports) if exp_ports else 0,
        'Avg Act Port Ret': np.mean(act_ports) if act_ports else 0,
        'Avg Pred Error': (np.mean(exp_ports) - np.mean(act_ports)) if exp_ports else 0,
        'Note': '',
    })

pred_df = pd.DataFrame(pred_rows)

# Format
disp_pred = pred_df.copy()
for c in ['Dir %', 'Avg Exp Port Ret', 'Avg Act Port Ret', 'Avg Pred Error']:
    disp_pred[c] = disp_pred[c].apply(lambda x: f'{x:+.2%}' if isinstance(x, float) else x)
disp_pred['MAE (test)'] = disp_pred['MAE (test)'].apply(lambda x: f'{x:.5f}')

print('-- Prediction Quality Comparison --')
print(disp_pred.to_string(index=False))

In [ ]:
# ── Expected vs Actual portfolio return scatter ──
arch_colors = {'cnn':'#2196F3','conv1d':'#2196F3','mlp':'#E91E63','rnn':'#FF9800','mixto':'#4CAF50'}

# Only models with detail data
models_with_detail = {t: d for t, d in models.items() if d['detail'] is not None and not d['detail'].empty}

if models_with_detail:
    fig, ax = plt.subplots(figsize=(8, 6))

    for tag, data in sorted(models_with_detail.items()):
        det = data['detail']
        arch = data['meta'].get('arch', '?')
        color = arch_colors.get(arch, '#888')

        windows = det.groupby('window')
        for _, grp in windows:
            exp_p = (grp['weight_eq'] * grp['exp_return']).sum() * 100
            act_p = (grp['weight_eq'] * grp['actual_return']).sum() * 100
            ax.scatter(exp_p, act_p, color=color, alpha=0.7, s=50,
                       edgecolors='white', lw=0.5)

        # Invisible marker for legend
        ax.scatter([], [], color=color, label=tag, s=50)

    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
            max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'k--', alpha=0.3, lw=1)
    ax.axhline(0, color='gray', alpha=0.3, lw=0.5)
    ax.axvline(0, color='gray', alpha=0.3, lw=0.5)

    ax.set_xlabel('Expected Portfolio Return (%, Strat A)', fontsize=11)
    ax.set_ylabel('Actual Portfolio Return (%)', fontsize=11)
    ax.set_title('Expected vs Actual Portfolio Returns — All Models, All Windows', fontsize=12)
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.savefig(os.path.join(COMP_DIR, 'comparison_exp_vs_actual.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No window detail data available for scatter plot.')

---
## <font color=#0099CC>**7. SAVE COMPARISON**</font>

In [ ]:
comp_df.to_csv(os.path.join(COMP_DIR, 'comparison_summary.csv'), index=False)
pred_df.to_csv(os.path.join(COMP_DIR, 'comparison_prediction_quality.csv'), index=False)

print('Comparison files saved to 11_Portfolio/:')
for f in sorted(os.listdir(COMP_DIR)):
    if f.startswith('comparison'):
        size = os.path.getsize(os.path.join(COMP_DIR, f))
        print(f'  {f}  ({size:,} bytes)')